# Studio 3 — your first night in Pyomo 🥫🍔

**OPIM 5641 · Business Decision Modeling · Dr. Dave Wanik, UConn**

So far we've solved LPs three ways: brute force, the graphical method, and Simplex by hand. Tonight
we hand the work to a solver. You write down the model, Pyomo builds it, and the solver does the
Simplex for you.

| | What | Who drives |
|---|---|---|
| **Part 1** | Warm-up: Flair's Furniture in Pyomo. You already know the answer (320 tables, 360 chairs, \$4,040). | together |
| **Part 2** | Stigler's diet: 77 foods, 9 nutrients, and the cheapest way to stay alive in 1939. | a solved example - run it and read it |
| **Part 3** | McDonald's: feed a football team for a day, then play with the rules. | **you** |
| **Part 4** | If there's time: sensitivity analysis. What is one more carpentry hour worth? | together |

The big idea tonight is the **constraint list**. Write a constraint once inside a loop, and it doesn't
care if there are 2 products or 77 foods.

## Import modules (download Pyomo)

You must do this every time! This installs Pyomo and the `cbc` solver in Colab.

In [ ]:
# import modules

# this makes beautiful plots using pylab (matplotlib)
%matplotlib inline
from pylab import * # * means import ALL NAME SPACES
import pandas as pd

# useful modules for downloading pyomo onto Colab
import shutil
import sys
import os.path

# install pyomo if it doesn't exist
if not shutil.which("pyomo"):
    !pip install -q pyomo
    assert(shutil.which("pyomo"))

# install the 'cbc' solver if it doesn't exist
if not (shutil.which("cbc") or os.path.isfile("cbc")):
    if "google.colab" in sys.modules:
        !apt-get install -y -qq coinor-cbc
    else:
        try:
            !conda install -c conda-forge coincbc
        except:
            pass

# make sure 'cbc' is the solver that we will invoke later on
assert(shutil.which("cbc") or os.path.isfile("cbc"))

# import ALL NAMESPACES (variables) from pyomo
from pyomo.environ import *

# Part 1 · Warm-up: Flair's Furniture in Pyomo

Same problem as Studio 2. Flair makes **tables** and **chairs**.

| | Tables | Chairs | Hours available |
|---|---|---|---|
| Carpentry | 3 | 4 | 2,400 |
| Painting | 2 | 1 | 1,000 |
| **Profit (USD)** | 7 | 5 | - |

Plus: **no more than 450 chairs**, and **at least 100 tables**.

$$Max\ Z = 7T + 5C$$

subject to:
* $3T + 4C \le 2400$ `carpentry`
* $2T + 1C \le 1000$ `painting`
* $C \le 450$ `chair limit`
* $T \ge 100$ `table minimum`
* $T, C \ge 0$ `non-negativity`

## 1a · One constraint at a time

This is the general framework: **model → variables → objective → constraints → solve → read the
results.** Every Pyomo model you ever write has these same pieces.

In [ ]:
# you will always need to write this code
model = ConcreteModel()

# declare decision variables
model.tables = Var(domain=NonNegativeReals)
model.chairs = Var(domain=NonNegativeReals)

# declare objective
model.profit = Objective(
                      expr = 7*model.tables + 5*model.chairs, # values come from the table
                      sense = maximize)

# write the constraints one by one, don't forget LHS vs. RHS
# it's OK to have mixed constraints (LHS <= RHS vs. LHS >= RHS)!!!
model.constraint1 = Constraint(expr = 3*model.tables + 4*model.chairs <= 2400) # carpentry hours
model.constraint2 = Constraint(expr = 2*model.tables + 1*model.chairs <= 1000) # painting hours
model.constraint3 = Constraint(expr = model.chairs <= 450) # chair limit
model.constraint4 = Constraint(expr = model.tables >= 100) # table minimum

# show the model you've created
model.pprint()

In [ ]:
#            solver          path the solver
SolverFactory('cbc', executable='/usr/bin/cbc').solve(model).write()

Look for `Termination condition: optimal`. If you see it, you solved! If not, you probably have a
bug (a typo in a constraint, the wrong sense, etc.).

Now read the results. **Don't forget the `()` after everything** - lots of students forget this and
get a bug!

In [ ]:
print("Profit = ", model.profit(), " USD")
print("Tables = ", model.tables(), " units")
print("Chairs = ", model.chairs(), " units")
print()

# the left-hand side of each constraint - how much did we actually use?
print("Carpentry = ", model.constraint1(), " hours (of 2400)")
print("Painting = ", model.constraint2(), " hours (of 1000)")
print("Chairs = ", model.constraint3(), " units (limit 450)")
print("Tables = ", model.constraint4(), " units (minimum 100)")
print()

# slack = what is left over (0 means it's all used up!)
print("Carpentry slack = ", model.constraint1.uslack(), " hours")
print("Painting slack = ", model.constraint2.uslack(), " hours")
print("Chair slack = ", model.constraint3.uslack(), " units")

**320 tables, 360 chairs, \$4,040.** Same answer as brute force, the graphical method, and
Simplex. Fourth way, same answer.

Which constraints are **binding** (LHS = RHS, no slack)? Carpentry and painting: every hour is used
up. Those are the **bottlenecks**, and that's the corner where those two lines crossed on your graph.
The chair limit has 90 units of slack - the same 90 you found for $s_3$ on your Simplex math check.

## 1b · Same model, soft-coded, with a constraint list

Writing constraints one by one is fine for 4 constraints. It's not fine for 400. So we do two things
we already know from brute force:

1. **Soft-code the data** in dictionaries.
2. **Write each kind of constraint once, inside a loop**, and add it to a `ConstraintList()`.

The decision variables become **one indexed variable**, `model.x[...]`, with one entry per product.

In [ ]:
# soft-code the data
products = ["tables", "chairs"]

profit = {
    "tables" : 7,
    "chairs" : 5
}

# hours each product needs in each department
hours = {
    "carpentry" : {"tables" : 3, "chairs" : 4},
    "painting"  : {"tables" : 2, "chairs" : 1}
}

# hours available this month
total_time = {
    "carpentry" : 2400,
    "painting"  : 1000
}

In [ ]:
# declare the model
# ONE indexed variable over products: model.x = Var(products, domain=NonNegativeReals)
# objective: obj_expr = 0, then loop over products and += profit[product] * model.x[product]
# model.constraints = ConstraintList()
# loop over departments: build dept_expr, then model.constraints.add(dept_expr <= total_time[dept])
# then add the chair limit and the table minimum
# model.pprint()
# YOUR CODE HERE 🛠️


In [ ]:
# solve it with cbc, then print the profit and each product (don't forget the () )
# YOUR CODE HERE 🛠️


Same answer. But look at what changed: **the constraint code no longer knows how many products or
departments there are.** Add a desk to `products` and a shipping department to `hours`, and the loop
handles it. That's what we need for the next problem, which has 77 products.

# Part 2 · Stigler's diet 🥫 (a solved example)

In 1945, the economist **George Stigler** asked a simple question: what is the cheapest diet that
gives a person every nutrient they need for a year? He had **77 foods** and **9 nutrients**, with 1939
prices.

He couldn't solve it exactly (Simplex hadn't been invented yet), so he used clever trial and error
and got **\$39.93 a year**. A couple of years later, it became one of the first big problems ever
solved with the Simplex method: nine clerks with desk calculators needed about 120 person-days. The
true optimum turned out to be a little lower than Stigler's guess.

Tonight it takes your laptop less than a second.

**This part is already solved for you.** Run each cell and read it carefully, because in Part 3 you
will build the same model yourself for McDonald's.

**Before you run anything, guess:** (1) the cheapest cost per year, in 1939 dollars, and (2) how many
of the 77 foods end up in the diet. Write both guesses down.

**A twist in the data.** Each number in Stigler's table is **how much of a nutrient you get for \$1
of that food** (1939 prices). So the decision variable is **how many dollars per day to spend on each
food**, and the cost of the diet is just the total of those dollars.

$$Min\ Z = \sum_{f} x_f$$

subject to, for every nutrient $n$:

$$\sum_{f} (\text{amount of nutrient } n \text{ per dollar of food } f) \cdot x_f \ge \text{daily minimum}_n$$

$$x_f \ge 0$$

That's a **covering** model: minimize cost, cover every requirement (`>=`).

The data is the same as the Google OR-Tools Stigler example. Let's read it from the CSV.

In [ ]:
# read the data
foods_df = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio3/stigler_foods.csv')
req_df = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio3/stigler_requirements.csv')

print(len(foods_df), "foods")
foods_df.head()

In [ ]:
# the daily minimums - calories are in THOUSANDS (3 = 3,000 calories)
req_df

## Soft-code the data into dictionaries

Same idea as Flair's: one dictionary per piece of the model. Loops only, so you can see every step.

In [ ]:
# the list of foods and the list of nutrients
foods = list(foods_df['food'])
nutrients = list(req_df['nutrient'])

# daily minimum for each nutrient, plus a readable label
minimum = {}
label = {}
for i in range(len(req_df)):
    nutrient = req_df.loc[i, 'nutrient']
    minimum[nutrient] = float(req_df.loc[i, 'daily_minimum'])
    label[nutrient] = req_df.loc[i, 'label']

# content[food][nutrient] = how much of that nutrient you get for $1 of that food
content = {}
for i in range(len(foods_df)):
    food = foods_df.loc[i, 'food']
    content[food] = {}
    for nutrient in nutrients:
        content[food][nutrient] = float(foods_df.loc[i, nutrient])

print(minimum)
print()
print("Wheat Flour:", content['Wheat Flour (Enriched)'])

## Build the model

Look how short this is. **77 decision variables and 9 constraints**, and the constraint code is the
same shape as Flair's department loop.

In [ ]:
# declare the model
model = ConcreteModel()

# dollars per day to spend on each food (77 of them!)
model.x = Var(foods, domain=NonNegativeReals)

# objective: the total dollars per day
obj_expr = 0
for food in foods:
    obj_expr += model.x[food]

model.cost = Objective(
                      expr = obj_expr,
                      sense = minimize)

# one constraint per nutrient (this loop runs 9 times)
model.constraints = ConstraintList()
for nutrient in nutrients:
    nutrient_expr = 0
    for food in foods:
        nutrient_expr += content[food][nutrient] * model.x[food]
    model.constraints.add(nutrient_expr >= minimum[nutrient])

# careful - model.pprint() is LONG with 77 foods, so just count instead
print("Number of variables =", len(model.x))
print("Number of constraints =", len(model.constraints))

In [ ]:
SolverFactory('cbc', executable='/usr/bin/cbc').solve(model).write()

## Read the results

Same printout as the Google OR-Tools example: what to buy for the year, the annual price, and the
nutrients per day against each minimum.

In [ ]:
# what to buy for a whole year (only the foods we actually buy)
print("Annual Foods:")
for food in foods:
    if model.x[food]() > 0.000001:
        print(food, ": $", round(365 * model.x[food](), 2))

print()
print("Optimal annual price: $", round(365 * model.cost(), 4))

# the constraint list is numbered 1, 2, 3 ... in the order we added them
print()
print("Nutrients per day:")
k = 1
for nutrient in nutrients:
    print(label[nutrient], ":", round(model.constraints[k](), 2), "(min", minimum[nutrient], ")")
    k += 1

**\$39.66 a year** - about 11 cents a day. Stigler's trial-and-error guess was \$39.93, so he was
off by about 27 cents. Not bad for doing it by hand! How close was *your* guess?

The winning diet: **wheat flour, beef liver, cabbage, spinach, and navy beans.** For a year.

Which nutrients are **binding** (exactly at the minimum)? Those are the ones driving the cost. The ones
with room to spare (protein, iron...) come along for free.

**Talk with your partner:**
1. Out of 77 foods, the solver picked only 5. Why so few? (Hint: think about corner points. With 9
   constraints, how many variables can be non-zero at a corner?)
2. Would you eat this diet? What's missing from the model?

# Part 3 · Your turn: feed a football team at McDonald's 🏈🍔

The team bus pulls into McDonald's. **53 players**, one day of meals, and the athletic department is
paying. What's the cheapest order that keeps everyone fueled?

It's Stigler's diet again, with **three changes**:

| | Stigler | McDonald's |
|---|---|---|
| **Decision variable** | dollars per day on each food | **how many of each menu item** to order for the whole team |
| **Objective** | add up the dollars | add up **price × how many** |
| **Rules** | every nutrient has a minimum (`>=`) | some are minimums (`>=`), some are **maximums** (`<=`), and they are **per player**, so multiply by the team size |

Plus one new rule for variety: **nobody gets more than 4 of the same item.**

The nutrition facts come from McDonald's published nutrition data (the Kaggle *Nutrition Facts for
McDonald's Menu* dataset). The prices are **approximate** US prices; change them to match your local
McDonald's if you like.

**Before you solve it, guess:** what's the cheapest bill per player? Write it down.

In [ ]:
# read the data
menu_df = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio3/mcdonalds_menu.csv')
rules_df = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio3/mcdonalds_rules.csv')

menu_df

In [ ]:
# the rules, per player per day - notice calories has BOTH a min and a max
rules_df

## Soft-code the data into dictionaries

This part is done for you. Read it, run it, and look at what's in each dictionary.

In [ ]:
# the menu items - every column after 'price' is a nutrient
items = list(menu_df['item'])
mc_nutrients = list(menu_df.columns[3:])

# price[item] and nutrition[item][nutrient] (per ONE item)
price = {}
nutrition = {}
for i in range(len(menu_df)):
    item = menu_df.loc[i, 'item']
    price[item] = float(menu_df.loc[i, 'price'])
    nutrition[item] = {}
    for nutrient in mc_nutrients:
        nutrition[item][nutrient] = float(menu_df.loc[i, nutrient])

# the rules per player: one dictionary of minimums, one of maximums
player_min = {}
player_max = {}
mc_label = {}
for i in range(len(rules_df)):
    nutrient = rules_df.loc[i, 'nutrient']
    mc_label[nutrient] = rules_df.loc[i, 'label']
    if rules_df.loc[i, 'rule'] == 'min':
        player_min[nutrient] = int(rules_df.loc[i, 'per_player'])
    else:
        player_max[nutrient] = int(rules_df.loc[i, 'per_player'])

print("minimums per player:", player_min)
print("maximums per player:", player_max)
print()
print("Big Mac: $", price['Big Mac'], nutrition['Big Mac'])

# an empty scoreboard - we fill this in as we play
scoreboard = {}

## Build the model (this one is yours)

Scroll up to the Stigler model and use it as your recipe. The knobs at the top and the house rules at
the bottom are there for the game we play next - for now, just write the model in the middle.

In [ ]:
# ---------- THE KNOBS (change these, then re-run this cell and the next one) ----------
round_name = "1 base"
team_size = 53          # players on the roster
max_of_one_item = 4     # nobody gets more than 4 of the same item

# a fresh copy of the rules from the CSV, so every round starts clean
low = dict(player_min)
high = dict(player_max)

# ---------- THE MODEL ----------
# 1. declare the model
# 2. model.x = Var(items, domain=NonNegativeReals)     <- how many of each item for the whole team
# 3. objective: obj_expr = 0, loop over items, += price[item] * model.x[item], sense = minimize
#    (call it model.cost)
# 4. model.constraints = ConstraintList()
# 5. the minimums: for nutrient in low:
#        build nutrient_expr from nutrition[item][nutrient] * model.x[item]
#        model.constraints.add(nutrient_expr >= low[nutrient] * team_size)
# 6. the maximums: same loop over high, but <=
# 7. variety: for item in items:  model.x[item] <= max_of_one_item * team_size
# YOUR CODE HERE 🛠️


# ---------- HOUSE RULES (leave this empty for round 1) ----------


print("Number of variables =", len(model.x))
print("Number of constraints =", len(model.constraints))

You should see **20 variables and 31 constraints** (8 minimums + 3 maximums + 20 variety rules).

## Solve it and see the order

This cell is done for you: it solves the model, prints the order, checks the nutrition per player,
draws the order, and writes the result on the scoreboard.

In [ ]:
results = SolverFactory('cbc', executable='/usr/bin/cbc').solve(model)

if results.solver.termination_condition == TerminationCondition.optimal:
    per_player = round(model.cost() / team_size, 2)
    print("ROUND:", round_name)
    print("Total bill = $", round(model.cost(), 2))
    print("Per player = $", per_player)

    print()
    print("The order:")
    names = []
    counts = []
    for item in items:
        if model.x[item]() > 0.000001:
            print(item, ":", round(model.x[item](), 1), "total (", round(model.x[item]() / team_size, 2), "per player )")
            names.append(item)
            counts.append(model.x[item]() / team_size)

    print()
    print("Per player per day:")
    for nutrient in mc_nutrients:
        total = 0
        for item in items:
            total += nutrition[item][nutrient] * model.x[item]()
        rule_text = ""
        if nutrient in low:
            rule_text += " min " + str(low[nutrient])
        if nutrient in high:
            rule_text += " max " + str(high[nutrient])
        print(mc_label[nutrient], ":", round(total / team_size, 1), "(" + rule_text + " )")

    # write it on the scoreboard
    scoreboard[round_name] = per_player

    # a picture of the order
    barh(names, counts, color='darkorange')
    xlabel('how many per player')
    title(round_name + ':  $%.2f per player' % per_player)
    tight_layout()
    savefig('fig_team_order.png')
    show()
else:
    print("ROUND:", round_name)
    print("INFEASIBLE! No order can satisfy every rule at the same time.")
    print("Loosen a rule and try again.")

How close was your guess?

Look at the per-player numbers. Which rules are **binding**? Those are what's driving the bill.
(And yes, that's a lot of apple pie.)

## Play with the constraints 🎮

Now the fun part. For each round below: **change the knobs or add a house rule in the model cell,
re-run that cell, then re-run the solve cell.** Give each round its own `round_name` so the scoreboard
keeps them all.

| Round | What to change | Where |
|---|---|---|
| **1 base** | nothing - you just did it | |
| **2 no variety rule** | `max_of_one_item = 100` | knobs |
| **3 coach's order** | everybody gets a Big Mac and a medium fries | house rules |
| **4 team doctor** | cut the sodium limit to 5,000 mg: `high['sodium'] = 5000` | knobs (after the `high = ...` line) |
| **5 break it** | the FDA says 2,300 mg of sodium a day: `high['sodium'] = 2300` | knobs |
| **6 your call** | make up your own rule - an order you would actually eat | anywhere |

House rules are just more constraints. Copy, paste and edit these:

```python
# everybody gets a Big Mac and a medium fries (at least 1 per player)
model.constraints.add(model.x['Big Mac'] >= 1 * team_size)
model.constraints.add(model.x['Medium French Fries'] >= 1 * team_size)

# nobody eats the fish
model.constraints.add(model.x['Filet-O-Fish'] == 0)

# linemen need more protein
low['protein'] = 200        # this one goes up in the knobs, right after  low = dict(player_min)
```

Remember to put the knobs back (`max_of_one_item = 4`) and clear out old house rules between rounds.

In [ ]:
# the scoreboard so far
print(scoreboard)

## Your deliverable: the team order card 📋

When you've played your rounds, run the cell below to draw your scoreboard, then answer the three
questions underneath it. That's what you turn in tonight.

In [ ]:
# draw the scoreboard
round_names = []
dollars = []
for name in scoreboard:
    round_names.append(name)
    dollars.append(scoreboard[name])

barh(round_names, dollars, color='steelblue')
xlabel('$ per player')
title('Feed the team: what did each rule cost?')
gca().invert_yaxis()
tight_layout()
savefig('fig_scoreboard.png')
show()

**Double-click this cell and answer:**

1. **Round 2:** what did the cheapest order look like with no variety rule, and how much did variety
   cost per player?

   *your answer*

2. **Rounds 4 and 5:** what did cutting the sodium to 5,000 mg cost per player? What happened at
   2,300 mg, and what does that tell you about a day of McDonald's?

   *your answer*

3. **Round 6:** what rule did you add, and what did it cost the team? Was it worth it?

   *your answer*

**One more thing to think about:** you can't order 1.73 hamburgers per player. What would you change
in the model to get whole numbers? (That's integer programming - coming in Module 5.)

# Part 4 · If there's time: sensitivity analysis

Back to Flair's. Remember the bottlenecks: **all 2,400 carpentry hours were used up.** So a savvy
analyst asks: *what if I could get a few more carpentry hours? How much is one more hour actually
worth?* That number is the **shadow price**.

Here is a very transparent way to find it: **take your model and stick it in a for loop.** The thing
you change every time is the right-hand side of the binding constraint.

Buffer out a little: go backwards from 2,400, and explore well beyond it.

In [ ]:
myHours = np.arange(1200, 3300, 100)
print(myHours)

In [ ]:
# myResults = pd.DataFrame()
# for a in myHours:
#     rebuild Flair's model (one constraint at a time is fine), but carpentry is <= a instead of <= 2400
#     solve it (leave off the .write() or you get 21 printouts!)
#     store [a, profit, tables, chairs] as a row with pd.concat
# rename the columns: Carpentry Hours, Profit, Tables, Chairs
# YOUR CODE HERE 🛠️


## Product mix

Read the table before you plot it. As the carpentry hours go up, profit goes up - and the **mix**
changes. Let's plot the product mix as a function of carpentry hours.

In [ ]:
# drop the Profit column and plot Tables and Chairs against Carpentry Hours
# YOUR CODE HERE 🛠️


With very few carpentry hours we only make tables (they earn more per carpentry hour). As hours
are added we trade tables for chairs, until we hit the 450-chair limit and the mix stops changing.

## Profit and diminishing returns

Total profit is misleading - of course you make more money as you get more hours. The question is how
much **each additional hour** is worth. Subtract one row from the next and divide by the step.

In [ ]:
# add a column: the change in profit divided by the change in carpentry hours
# YOUR CODE HERE 🛠️


In [ ]:
# plot ProfitDiffOneHour against Carpentry Hours
# YOUR CODE HERE 🛠️


Read the plot as a staircase:

- **Up to 1,500 hours:** carpentry is so scarce we only make tables, and each extra hour is worth
  about **\$2.33** (one more third of a table, at \$7 each).
- **From 1,500 to about 2,600 hours:** each extra hour is worth exactly **\$0.60**. That's the
  **shadow price** of carpentry at our optimum.
- **Beyond that:** we hit the 450-chair limit, and more carpentry is worth **\$0**.

Your carpentry hours start out like **diamonds** and turn into **sand**: once they're not scarce
anymore, they have no value, because the bottleneck has moved somewhere else in the shop.

**The business question:** suppose an extra carpentry hour costs Flair **\$0.50** in overtime. How many
carpentry hours would you advocate for? (Add hours while the shadow price is above the cost: go to
about 2,600 hours and stop.)

**Tie it back to Simplex:** if you took Flair's all the way to the final tableau, look at the bottom
row under the carpentry slack: **3/5 = 0.60**. The bottom row of the final tableau *is* the shadow
prices.

**On your own:**
1. Run the same loop on the painting hours. What is one more painting hour worth? (Check your final
   tableau again.)
2. **The price of salt.** Stick your McDonald's model in the same kind of loop and vary the sodium
   limit from 4,500 to 8,000 mg. What does each 500 mg of sodium save the team per player?

## Bottom line

- The Pyomo recipe never changes: **model → variables → objective → constraints → solve → read the
  results.**
- A **ConstraintList** plus a loop lets you write a constraint once and apply it to every department,
  every nutrient, every menu item. Two products or 77 foods, same code.
- **Soft-code the data** (dictionaries, or straight from a CSV), and the model doesn't change when the
  data does. Stigler's diet and the McDonald's order are the same model with different CSVs.
- **Binding** constraints are the bottlenecks, and a for loop around your model tells you exactly
  what loosening them is worth.

**Before you leave:** File → Save a copy in GitHub (your scoreboard and your three answers go with it).